# Rewrite the joined timeseries tables

`fcst_joined_timeseries` and `sim_joined_timeseries` have never been compacted. Until the binpack
fallback landed in `routine_table_maintenance.py`, `get_rewrite_settings` skipped any table without a
`write.sort-order` or `write.target-file-size-bytes` property, which was every table except
`primary_timeseries` and `secondary_timeseries`.

Run this once to do the first pass by hand. After that the nightly maintenance flow keeps up
incrementally.

**Why not just let the nightly job do it?** The `rewrite_data_files` task is `timeout_seconds = 45 * 60`
with `retries = 1`. A ~67 GB first pass may not finish inside that window, and a timeout would retry from
scratch and time out again. The calls below set `partial-progress.enabled` so work commits in batches and
a timeout keeps whatever completed.

**Order matters for `sim_joined_timeseries`.** Re-run
`warehouse/remote/03_preprocessing/simulation/01_create_joined_timeseries.ipynb` **before** this notebook.
That notebook sets `partition_by` and `write_ordered_by`; `rewrite_data_files` cannot change a partition
spec, so rewriting first would just compact the table into its current single unpartitioned layout. The
cell below guards against this.

You must have write permissions to the data warehouse. Expect tens of minutes — this is a full rewrite of
roughly 67 GB.

In [ ]:
from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=20,
    executor_memory="32g",
    executor_cores=6,
    aws_profile="default",
)

## Before

In [ ]:
def report(table_name):
    """Print file and partition layout so the rewrite can be measured."""
    f = spark.sql(f"""
        SELECT count(*) AS files,
               round(sum(file_size_in_bytes) / 1e9, 2) AS gb,
               round(avg(file_size_in_bytes) / 1e6, 1) AS avg_mb
        FROM iceberg.teehr.{table_name}.files
    """).collect()[0]
    p = spark.sql(f"""
        SELECT count(*) AS partitions,
               round(avg(file_count), 1) AS files_per_partition,
               max(file_count) AS max_files
        FROM iceberg.teehr.{table_name}.partitions
    """).collect()[0]
    props = {
        r["key"]: r["value"]
        for r in spark.sql(f"SHOW TBLPROPERTIES iceberg.teehr.{table_name}").collect()
    }
    print(table_name)
    print(f"  {f['files']} files, {f['gb']} GB, {f['avg_mb']} MB avg")
    print(f"  {p['partitions']} partitions, {p['files_per_partition']} files/partition "
          f"(max {p['max_files']})")
    # `sort-order` is Iceberg's reflection of a real sort order. A `write.sort-order`
    # key is the inert custom property from migrations 0002/0006 and means nothing.
    print(f"  sort order: {props.get('sort-order', '(none declared)')}")


for table in ["fcst_joined_timeseries", "sim_joined_timeseries"]:
    report(table)

## fcst_joined_timeseries

Partitioned by `months(value_time), configuration_name` and sorted by
`primary_location_id, secondary_location_id, reference_time, value_time` — both set by
`prefect-workflows/workflows/metrics/utils/joined_forecast_utils.py`. Nothing to change here; it just
needs compacting.

No `sort_order` argument is passed: `strategy => 'sort'` uses the table's declared sort order. Passing one
explicitly (as `03_rewrite_timeseries.ipynb` does) overrides it and can silently sort by the wrong
columns.

`rewrite-all` is set because the data has never been sorted. Without it, partitions holding fewer than
`min-input-files` (5) files would be left untouched and stay unsorted. Drop it on later runs.

In [ ]:
%%time
spark.sql("""
    CALL iceberg.system.rewrite_data_files(
        table => 'teehr.fcst_joined_timeseries',
        strategy => 'sort',
        options => map(
            'rewrite-all', 'true',
            'partial-progress.enabled', 'true',
            'partial-progress.max-commits', '10',
            'max-concurrent-file-group-rewrites', '10'
        )
    )
""").show()

## sim_joined_timeseries

Requires `01_create_joined_timeseries.ipynb` to have been re-run first — see the note at the top. The
guard below fails fast rather than compacting 14.7 GB into the wrong layout.

In [ ]:
props = {
    r["key"]: r["value"]
    for r in spark.sql("SHOW TBLPROPERTIES iceberg.teehr.sim_joined_timeseries").collect()
}
if "sort-order" not in props:
    raise RuntimeError(
        "sim_joined_timeseries has no declared sort order. Re-run "
        "warehouse/remote/03_preprocessing/simulation/01_create_joined_timeseries.ipynb "
        "first - it sets partition_by and write_ordered_by. Rewriting now would compact "
        "the table into its current single unpartitioned spec."
    )
print(f"sort order declared: {props['sort-order']}")

In [ ]:
%%time
spark.sql("""
    CALL iceberg.system.rewrite_data_files(
        table => 'teehr.sim_joined_timeseries',
        strategy => 'sort',
        options => map(
            'rewrite-all', 'true',
            'partial-progress.enabled', 'true',
            'partial-progress.max-commits', '10',
            'max-concurrent-file-group-rewrites', '10'
        )
    )
""").show()

## After

Expect far fewer, larger files. Snapshot and orphan-file cleanup is left to the nightly
`routine_table_maintenance` flow, which expires snapshots older than 7 days — the pre-rewrite files stay
on S3 until then.

In [ ]:
for table in ["fcst_joined_timeseries", "sim_joined_timeseries"]:
    report(table)

In [ ]:
spark.stop()